In [ ]:
#   numpy(넘파이)는 숫자 여러 개를 '배열(array)'로 묶어 한 번에 계산하게 해 주는 라이브러리입니다.
#   예) [160, 170, 180]에 모두 2를 더하는 계산을 한 줄로 할 수 있습니다.
#   이 노트북에서는 데이터 준비/정규화/학습 전 확인 같은 'Numpy 흐름'에 사용합니다.
import numpy as np

#   torch는 Pytorch 라이브러리입니다.
#   이번 노트북에서는 '자동 미분(autograd)' 기능을 쓰기 위해 사용합니다.
#   자동 미분이란, 사람이 미분 공식을 직접 적지 않아도
#   Pytorch가 계산 과정을 따라가며 기울기(미분값)를 대신 계산해 주는 기능입니다.
import torch

from math import e

In [ ]:
#   ========================================
#   1. 입력값 X와 정답 y 준비   (기존 NumPy 흐름 그대로 유지)
#   ========================================

#   X: 입력값입니다. 여기서는 사람의 키(cm)를 사용합니다.
#       np.array([...]) 는 여러 숫자를 하나의 NumPy 배열로 묶는 것입니다.
X = np.array([160, 170, 180, 190])

#   y: 정답값입니다. 0은 농구선수 아님, 1은 농구선수입니다.
#       X와 y는 순서대로 짝지어집니다. 즉 키 160 → 정답 0, 키 190 → 정답 1.
y = np.array([0, 0, 1, 1])

print('입력값 X:', X)
print('정답값 y:', y)

입력값 X: [160 170 180 190]
정답값 y: [0 0 1 1]


In [ ]:
#   ========================================
#   2. 입력값 정규화    (기존 NumPy 흐름 그대로 유지)
#   ========================================

#   평균(mean)과 표준편차(std)를 계산합니다.
#   - 평균      : 데이터의 중심(가운데가 되는 값)
#   - 표준편차   : 데이터가 평균에서 얼마나 넓게 퍼져있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

#   정규화 공식: (원본값 - 평균) / 표준편차
#   입력값의 범위를 0 근처로 비슷하게 맞추면 학습이 더 안정적으로 진행됩니다.
#   주의: 실제 학습에는 원래 키 X가 아니라, 정규화된 입력값 X_norm을 사용합니다.
#       (X_mean, X_std 는 나중에 '새 입력값 예측'에서도 똑같이 다시 씁니다.)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean: ', X_mean)
print('입력값 표준편차 X_std: ', X_std)
print('정규화된 입력값 X_norm: ', X_norm)

입력값 평균 X_mean:  175.0
입력값 표준편차 X_std:  11.180339887498949
정규화된 입력값 X_norm:  [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [ ]:
#   ========================================
#   2-1. X_norm 과 y 를 PyTorch tensor 로 변환
#   ========================================

#   PyTorch의 자동 미분은 'tensor(텐서)'라는 자료형 위에서 동작합니다.
#   tensor란, NumPy 배열과 거의 똑같이 생긴 Pytorch 전용 숫자 묶음입니다.
#   단, tensor는 '어떻게 계산되었는지'를 기억할 수 있어서 자동 미분이 가능합니다.
#
#   그래서 학습에 상요할 입력값(X_norm)과 정답(y)을 먼저 tensor로 바꿔 둡니다.
#
#   주의(햇갈리기 쉬움)
#       x       = 원래 키(cm)
#       X_norm  = 정규화된 입력값 ← 학습에는 이 값을 사용합니다.
#   따라서 아래에서도 X가 아니라 X_norm을 tensor로 변환합니다.

#   dtype = torch.float32: 소수점 계산(미분)을 위해 실수(float) 형식으로 만듭니다.
X_norm_tensor = torch.tensor(X_norm, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype = torch.float32)

print('학습용 입력 tensor X_norm_tensor: ', X_norm_tensor)
print('학습용 정답 tensor y_tensor: ', y_tensor)

학습용 입력 tensor X_norm_tensor:  tensor([-1.3416, -0.4472,  0.4472,  1.3416])
학습용 정답 tensor y_tensor:  tensor([0., 0., 1., 1.])


In [ ]:
#   ========================================
#   3. 가중치 a와 편향 b 초기값 설정    (requires_grad=True인 tensor)
#   ========================================

#   a는 가중치(weight), b는 편향(bias)입니다.
#   a는 원래 키(cm)가 아니라 정규화된 입력값 X_norm에 곱해지는 값입니다.
#
#   requires_grad = True눈 Pytorch에게
#   "이 값에 대한 미분값(기울기)를 계산해 줘"라고 알려주는 설정입니다.
#
#   여기서 a와 b는 학습으로 계속 수정해 나갈 값이므로 requires_grad=True를 설정합니다.
#   (반대로 입력 X_norm_tensor와 정답 y_tensor에는 일 설정을 하지 않았습니다.
#   그 둘은 학습으로 바꾸는 값이 아니기 때문입니다.)
a = torch.tensor(0.1, dtype = torch.float32, requires_grad = True)
b = torch.tensor(0.0, dtype = torch.float32, requires_grad = True)

#   a, b는 tensor라서 그냥 출력하면 tensor(...) 형태로 나옵니다.
#   .item() 으로 숫자만 깔끔하게 꺼내서 출력합니다. (.item() 설명은 아래 셀에서 자세히)
print('초기 가중치 a: ', a.item())
print('초기 편향   b: ', b.item())

초기 가중치 a:  0.10000000149011612
초기 편향   b:  0.0


In [ ]:
#   ========================================
#   4. 시그모이드 함수 정의 (학습 전 NumPy 확인용)
#   ========================================

#   시그모이드 함수는 선형 계산값 H(x)를 0~1 사이의 예측 확률 z로 변환합니다.
#   아래 함수는 '학습 전 예측/Cost 확인' (NumPy 흐름)에서만 사용합니다.
#   실제 학습 루프 안에서는 PyTorch의 torch.sigmoid 를 사용합니다.
def sigmoid(H):
    return 1 / (1 + e ** (-H))

In [ ]:
#   ========================================
#   5. 학습 전 예측 확인 (학습 전 NumPy 확인용)
#   ========================================

#   H(x) = a·X_norm + b
#   H(x)는 sigmoid에 들어가기 전의 선형 계산값입니다. (확률이 아닙니다.)
#   여기서는 학습 전 상태를 눈으로 확인하려고 NumPy로 잠깐 계산해 봅니다.
# 
#   a, b는 tensor이고 X_norm은 NumPy 배열입니다.
#   학습 전 '확인용' 계산이므로, a, b에서 .item()으로 숫자만 꺼내 NumPy와 함께 계산합니다.
#   (이 셀은 학습이 아니므로 미분 그래프와 연결될 필요가 없습니다.)
H = (a.item() * X_norm) + b.item()

#   z = sigmoid(H)
#   z는 농구선수일 예측 확률입니다. (0~1 사이) ← H(x)가 아니라 z가 확률입니다!
z = sigmoid(H)

print('학습 전 선형 계산값 H(x): ', H)
print('학습 전 시그모이드 예측 확률 z: ', z)

학습 전 선형 계산값 H(x):  [-0.13416408 -0.04472136  0.04472136  0.13416408]
학습 전 시그모이드 예측 확률 z:  [0.4665092  0.48882152 0.51117848 0.5334908 ]


In [ ]:
#   ========================================
#   6. 학습 전 비용(Cost) 계산 (기존 NumPy 흐름 그대로 유지)
#   ========================================

#   Binary Cross Entropy(BCE) 비용 함수입니다.  (BCE = 이진 교차 엔트로피)
#   흐름: 현재 a, b로 H(x)를 구하고 → sigmoid로 z를 구하고 → z와 y를 비교해 Cost를 계산합니다.
#
#   실제값 y가 1이면 z가 1에 가까울수록 Cost가 작아지고,
#   실제값 y가 0이면 z가 0에 가까울수록 Cost가 작아집니다.
#   즉 예측 z가 정답 y에 가까울수록 Cost가 작아지고, 멀수록 커집니다.
#
#   Costs = - {y * log(z) + (1 - y) * log(1 - z)}

#   log(0)을 방지하기 위해 z가 정확히 0 또는 1이 되지 않도록 제한합니다.
#   (log(0)이 문제인지는 다음 셀들의 마크다운에서 자세히 설명합니다.)
epsilon = 1e-7
z_safe = np.clip(z, epsilon, 1 - epsilon)

costs = - (y * np.log(z_safe) + (1 - y) * np.log(1 - z_safe))
mean_cost = np.mean(costs)

print('학습 전 각 샘플의 비용(Cost): ', costs)
print('학습 전 평균 비용(Cost): ', mean_cost)

학습 전 각 샘플의 비용(Cost):  [0.62831345 0.67103648 0.67103648 0.62831345]
학습 전 평균 비용(Cost):  0.6496749672265923


In [ ]:
#   ========================================
#   7. 학습 설정 (기존 흐름 그대로 유지)
#   ========================================

#   learning_rate(학습률)는 한 번에 a, b를 얼마나 크게 수정할지 정하는 값입니다.
#       - 너무 크면: 한 번에 너무 많이 움직여 학습이 출렁이거나 발산할 수 있습니다.
#       - 너무 작으면: 너무 조금씩 움직여 학습이 매우 느려집니다.
learning_rate = 0.1

#   epochs(에폭)는 전체 데이터를 몇 번 반복해서 학습할지 정하는 값입니다.
#   여기서는 같은 데이터로 1000번 반복 학습합니다.
epochs = 1000

In [ ]:
#   ========================================
#   8. 경사 하강법으로 학습 (미분 계산만 PyTorch autograd로 대체)
#   ========================================
#
#   한 번의 epoch(반복)에서 일어나는 단계:
#       1단계: H(x) = a·X_norm + b 계산
#       2단계: z = sigmoid(H) 계산
#       3단계: BCE Cost 계산
#       4단계: backward()로 Cost를 a, b에 대해 미분
#       5단계: a.grad, b.grad를 grad_a, grad_b로 저장   (출력용)
#       6단계: Gradient Descent로 a, b 업데이트
#       7단계: 다음 epoch를 위해 gradient 초기화
#       8단계: 학습 상태 출력

for epoch in range(epochs):
    
    #   ----- 1단계: 선형 계산값 H(x) -----
    #   H(x) = a·X_norm + b
    #   H(x)는 sigmoid에 들어가기 전의 선형 계산값입니다.
    #   H(x)는 확률이 아닙니다. (확률은 다음 단계의 z 입니다.)
    #   여기서는 학습용이므로 tensor인 a, b와 X_norm_tensor를 그대로 곱합니다.
    #   (.item()을 쓰지 않습니다. 그래야 계산 그래프가 연결되어 자동 미분이 가능합니다.)
    H = (a * X_norm_tensor) + b
    
    #   ----- 2단계: 예측 확률 z -----
    #   z는 H(x)를 sigmoid 함수에 넣어 얻은 예측 확률입니다.
    #   z는 0과 1 사이의 값입니다.
    #   z가 1에 가까우면 1(농구선수)일 가능성이 높고,
    #   z가 0에 가까우면 0(농구선수 아님)일 가능성이 높다고 해석합니다.
    z = torch.sigmoid(H)
    
    #   ----- 3단계: BCE Cost 직접 계산 -----    
    #   torch.clamp()는 log(0)을 방지하기 위한 안전장치입니다.
    #   z가 정확히 0 또는 1이 되면 log(0)이 되어 계산이 깨지므로,
    #   z를 epsilon ~ (1 - epsilon) 사이로 살짝 제한합니다.
    #   (수학 교재의 핵심 미분 흐름은 clamp를 제외한 기본 BCE + sigmoid 기준이며,
    #    그 핵심 결과는 dC/dH = z - y 입니다. clamp는 수치 안정성용 안전장치입니다.)
    z_safe = torch.clamp(z, epsilon, 1 - epsilon)
    
    #   Cost = - { y * log(z) + ( 1 - y ) * log(1 - z) }
    #   torch.mean()으로 4개 데이터의 Cost를 평균 내 하나의 평균 Cost를 만듭니다.
    costs = -(y_tensor * torch.log(z_safe) + (1 -y_tensor) * torch.log(1 - z_safe))
    mean_cost = torch.mean(costs)
    
    #   ----- 4단계: backward()로 자동 미분 -----    
    #   backward()는 평균 Cost에서 출발해 a, b까지 연결된 계산 과정(계산 그래프)을
    #   거꾸로 따라가면 미분값(기울기)을 계산합니다.
    #
    #   계산이 끝나면
    #       a.grad 에는 Cost를 a로 미분한 값이,
    #       b.grad 에는 Cost를 b로 미분한 값이 들어갑니다.
    #
    #   수학적으로는 다음과 같습니다.
    #       grad_a = 평균((z - y) × X_norm)     ← a.grad 가 이 값
    #       grad_b = 평균(z - y)                ← b.grad 가 이 값
    #   하지만 이 식을 직접 코드로 적지 않아도,
    #   backward()가 계산 그래프를 따라 같은 값을 자동으로 구해 줍니다.
    mean_cost.backward()

    #   ----- 5단계: a.grad, b.grad를 grad_a, grad_b로 저장 (출력용) -----    
    #   a.grad, b.grad는 tensor입니다. 출력/기록할 때는 .item()으로
    #   일반 숫자처럼 꺼내는 것이 초보자에게 이해하기 쉽습니다.
    #
    #   a.grad.item()은 기존 수식의 grad_a와 같은 의미이고,
    #   b.grad.item()은 기존 수식의 grad_b와 같은 의미입니다.
    #   (즉 a.grad 가 곧 grad_a 역할. 여기서는 출력 편의를 위해 숫자로만 꺼내 듭니다.)
    #
    #   주의: 7단계에서 gradient를 0으로 초기화하므로,
    #       반드시 초기화 '전에' 이렇게 값을 저장해 둬야 합니다.
    grad_a = a.grad.item()
    grad_b = b.grad.item()
    
    #   ----- 6단계: 경사 하강법으로 a, b 업데이트 -----    
    #   a, b를 직접 수정하는 작업은 미분 계산 기록(계산 그래프)에 포함되면 안 됩니다.
    #   우리가 원하는 건 '현재 Cost 기준으로 구한 a.grad, b.grad로 a, b를 고치는 것' 뿐이고,
    #   이 수정 동작 자체까지 미분 대상으로 기록할 필요는 없기 때문입니다.
    #
    #   이 부분은 기존 Gradient Descent 식과 같습니다.
    #       a = a - learning_rate × grad_a  (여기서 a.grad가 grad_a 역할)
    #       b = b - learning_rate × grad_b  (여기서 b.grad가 grad_b 역할)
    #   즉, Cost가 줄어드는 방향(기울기 반대 방향)으로 a, b를 조금씩 움직입니다.
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad

        #   ----- 7단계: 다음 epoch를 위해 gradient 초기화 -----        
        #   PyTorch는 기본적으로 미분값을 덮어쓰지 않고 계속 '누적(더하기)' 합니다.
        #   즉 backward()를 또 부르면 기존 grad 위에 새 grad가 더해집니다.
        #   따라서 다음 epoch에서 '새로 계산한 미분값만' 사용하려면
        #   기존 gradient를 0으로 초기화해야 합니다.
        #   zero_()에서 끝의 _ 는 새 tensor를 만드는 게 아니라 기존 tensor 값을 직접 0으로 바꾼다는 뜻입니다.
        a.grad.zero_()
        b.grad.zero_()
        
        #   ----- 8단계: 학습 상태 출력 -----        
        #   tensor가 그대로 출력되지 않도록 .item()으로 숫자만 꺼내 출력합니다.
        #   100 epoch마다 한 번씩, 그리고 마지막 epoch에서 출력합니다.
        if epoch % 100 == 0 or epoch == epochs - 1:
            print(
                f'Epoch {epoch}, '
                f'평균 비용(Cost): {mean_cost.item():.6f}, '
                f'grad_a: {grad_a:.6f}, '
                f'grad_b: {grad_b:.6f},'
                f'a: {a.item():.6f},'
                f'b: {b.item():.6f}'
            )

Epoch 0, 평균 비용(Cost): 0.649675, grad_a: -0.422248, grad_b: 0.000000,a: 0.142225,b: -0.000000
Epoch 100, 평균 비용(Cost): 0.198225, grad_a: -0.103535, grad_b: 0.000000,a: 2.069146,b: 0.000000
Epoch 200, 평균 비용(Cost): 0.133789, grad_a: -0.063040, grad_b: -0.000000,a: 2.860633,b: 0.000000
Epoch 300, 평균 비용(Cost): 0.104196, grad_a: -0.047123, grad_b: -0.000000,a: 3.401512,b: 0.000000
Epoch 400, 평균 비용(Cost): 0.086192, grad_a: -0.038251, grad_b: 0.000000,a: 3.824389,b: 0.000000
Epoch 500, 평균 비용(Cost): 0.073786, grad_a: -0.032440, grad_b: -0.000000,a: 4.175776,b: 0.000000
Epoch 600, 평균 비용(Cost): 0.064613, grad_a: -0.028273, grad_b: 0.000000,a: 4.478100,b: 0.000000
Epoch 700, 평균 비용(Cost): 0.057512, grad_a: -0.025108, grad_b: 0.000000,a: 4.744184,b: 0.000000
Epoch 800, 평균 비용(Cost): 0.051834, grad_a: -0.022607, grad_b: 0.000000,a: 4.982181,b: 0.000000
Epoch 900, 평균 비용(Cost): 0.047181, grad_a: -0.020573, grad_b: -0.000000,a: 5.197650,b: 0.000000
Epoch 999, 평균 비용(Cost): 0.043330, grad_a: -0.018898, grad

In [11]:
print(f'\n학습 완료 후의 최적값: a = {a.item()}, b = {b.item()}')


학습 완료 후의 최적값: a = 5.3927106857299805, b = 8.616920865733846e-08


In [ ]:
#   ========================================
#   9. 학습 완료 후 최종 가중치와 편향 확인
#   ========================================

#   학습된 a와 b는 정규화된 입력값 X_norm을 기준으로 학습된 값입니다.
#   (원래 키 X 기준이 아니라 X_norm 기준이라는 점을 기억하세요.)
#   a, b는 tensor이므로, item()으로 숫자만 꺼내서 출력합니다.
input_height = 185

input_norm = (input_height - X_mean) / X_std

with torch.no_grad():
    input_norm_tensor = torch.tensor(input_norm, dtype = torch.float32)
    H_input = (a * input_norm_tensor) + b
    probability = torch.sigmoid(H_input)
    
print(f'\n키가 {input_height}cm인 사람의 농구선수 확률: {probability.item():.4f}')

pred = 1 if probability.item() >= 0.5 else 0
if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')


키가 185cm인 사람의 농구선수 확률: 0.9920
판별 결과: 농구선수입니다.
